In [69]:
# ------------------------------------------------- #
# 1- Entrada -> dungeons com perfis, ex: A=4, C=2, M=1, I=3; Similiaridade cosseno; Filtragem baseada em item (nota -> PostFormAnswers)
# *[OK]
# 2- Entrada -> dungeons com perfis, ex: A=4, C=2, M=1, I=3; Correlação de Pearson; Filtragem baseada em item (nota -> PostFormAnswers)
# *[OK]
# 3- Entrada -> dungeons com perfis, ex: A=4, C=2, M=1, I=3; Similiaridade cosseno; Filtragem baseada no usuário (comportamento -> SuccessRate, TotalLostHealth_norm, EnemyKillRate, MapCompletion)
# *[OK]
# 4- Entrada -> dungeons com perfis, ex: A=4, C=2, M=1, I=3; Correlação de Pearson; Filtragem baseada no usuário (comportamento -> SuccessRate, TotalLostHealth_norm, EnemyKillRate, MapCompletion)
# *[OK]
# 14 - Hibrido cosseno
# 15 - Hibrido pearson
# ------------------------------------------------- #

# 5- Entrada -> nome das dungeons, ex: M:20T:30G:10E:40; Similiaridade cosseno; Filtragem baseada em item (nota -> PostFormAnswers)
# *[OK]
# 6- Entrada -> nome das dungeons, ex: M:20T:30G:10E:40; Correlação de Pearson; Filtragem baseada em item (nota -> PostFormAnswers)
# *[OK]
# 7- Entrada -> nome das dungeons, ex: M:20T:30G:10E:40; Similiaridade cosseno; Filtragem baseada no usuário (comportamento -> SuccessRate, TotalLostHealth_norm, EnemyKillRate, MapCompletion)
# *[OK]
# 8- Entrada -> nome das dungeons, ex: M:20T:30G:10E:40; Correlação de Pearson; Filtragem baseada no usuário (comportamento -> SuccessRate, TotalLostHealth_norm, EnemyKillRate, MapCompletion)
# *[OK]
# 16 - Hibrido cosseno
# 17 - Hibrido pearson
# ------------------------------------------------- #

# 9- Entrada -> granular, ex: Dugeon com esses dados: 20 salas, 4 chaves-fechaduras, 1.5 de linearidad; Similiaridade cosseno; Filtragem baseada em item (nota -> PostFormAnswers)
# *
# 10- Entrada -> granular, ex: Dugeon com esses dados: 20 salas, 4 chaves-fechaduras, 1.5 de linearidad; Correlação de Pearson; Filtragem baseada em item (nota -> PostFormAnswers)
# *
# 11 - Entrada -> granular, ex: Dugeon com esses dados: 20 salas, 4 chaves-fechaduras, 1.5 de linearidad; Similiaridade cosseno; Filtragem baseada no usuário (comportamento -> SuccessRate, TotalLostHealth_norm, EnemyKillRate, MapCompletion)
# *
# 12 - Entrada -> granular, ex: Dugeon com esses dados: 20 salas, 4 chaves-fechaduras, 1.5 de linearidad; Correlação de Pearson; Filtragem baseada no usuário (comportamento -> SuccessRate, TotalLostHealth_norm, EnemyKillRate, MapCompletion)
# *[OK]
# 18 - Hibrido cosseno
# 19 - Hibrido pearson
# ------------------------------------------------- #
# 13 - multiclasse
# ------------------------------------------------- #

# -> TODO: CORRIGIR/ATUALIZAR AS ANOTAÇÕES ACIMA


In [70]:
# -----------------------------
# bibliotecas -> TODO: Remover as que não uso
# -----------------------------
import random
import numpy as np
import json
import seaborn as sns
import pandas as pd
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.tree import DecisionTreeClassifier
from sklearn import svm
from sklearn.linear_model import SGDClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import cross_val_score
import os
import time
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, cohen_kappa_score
from sklearn.metrics import ConfusionMatrixDisplay
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

# -----------------------------
# GRUPOS DE MÉTODOS
# -----------------------------
ITEM_PROFILE_METHODS = {1,2,5,6}
USER_PROFILE_METHODS = {3,4,7,8}
STRUCTURE_ONLY_METHODS = {9,10}
STRUCTURE_PERF_METHODS = {11,12,13}
HYBRID_PREFORM_METHODS = {14,15}
HYBRID_LEVEL_METHODS = {16,17}
HYBRID_STRUCTURE_METHODS = {18,19}
UNNAMED = {20,21}


# -----------------------------
# PREPARANDO O DATASET
# -----------------------------

def initialize_data_frame():
    pd.set_option('display.max_columns', None)
    pd.set_option('display.width', 2000)
    sns.set(rc={'figure.figsize':(19.20, 10.80)})


def get_dungeon_data_frame():
    with open('backup.json') as data_file:
        json_data = json.load(data_file)
    return pd.DataFrame(json_data['__collections__']['dungeons']).T


def get_user_data_frame():
    with open('backup.json') as data_file:
        json_data = json.load(data_file)
    return pd.DataFrame(json_data['__collections__']['users']).T


def drop_invalid_rows(_data_frame):
    _data_frame.drop(_data_frame[_data_frame['UniqueRoomsEntered'] == 0].index, inplace=True)
    return _data_frame


def drop_unused_columns(_data_frame):
    _data_frame.drop('__collections__', inplace=True, axis=1)
    _data_frame = _data_frame.drop(list(_data_frame.filter(regex='^Total.*Quests')), axis=1)
    _data_frame.drop('CompletedQuests', inplace=True, axis=1)
    _data_frame.drop('CompletedKillQuests', inplace=True, axis=1)
    _data_frame.rename(columns={'CompletedDamageQuests': 'CompletedKillQuests'}, inplace=True)
    _data_frame.drop('TreasuresCollected', inplace=True, axis=1)
    _data_frame.drop('TotalNpcs', inplace=True, axis=1)
    _data_frame.drop('WeaponName', inplace=True, axis=1)
    _data_frame.drop('HeatMap', inplace=True, axis=1)
    return _data_frame


def get_percentages(_data_frame):
    _data_frame['LockUsage'] = _data_frame['LocksOpened'].div(_data_frame['TotalLocks'].values)*100
    _data_frame['KeyCollection'] = _data_frame['KeysCollected'].div(_data_frame['TotalKeys'].values)*100
    _data_frame['MapCompletion'] = _data_frame['UniqueRoomsEntered'].div(_data_frame['TotalRooms'].values)*100
    _data_frame['RoomRevisitRate'] = _data_frame['RoomsEntered'].div(_data_frame['TotalRooms'].values)*100
    _data_frame['EnemyKillRate'] = _data_frame['EnemiesKilled'].div(_data_frame['TotalEnemies'].values)*100
    _data_frame['TotalAttempts'] = _data_frame['TotalWins'] + _data_frame['TotalDeaths'].values
    _data_frame['TotalAttempts'] = _data_frame['TotalAttempts'].astype(float)
    _data_frame['TotalWins'] = _data_frame['TotalWins'].astype(float)
    _data_frame['Success Rate'] = _data_frame['TotalAttempts'].div(_data_frame['TotalWins'].values)*100
    _data_frame.replace([np.inf, -np.inf], 0, inplace=True)
    return _data_frame


def drop_processed_columns(_data_frame):
    _data_frame.drop('LocksOpened', inplace=True, axis=1)
    _data_frame.drop('TotalLocks', inplace=True, axis=1)
    _data_frame.drop('KeysCollected', inplace=True, axis=1)
    _data_frame.drop('TotalKeys', inplace=True, axis=1)
    _data_frame.drop('EnemiesKilled', inplace=True, axis=1)
    _data_frame.drop('TotalEnemies', inplace=True, axis=1)
    _data_frame.drop('TotalAttempts', inplace=True, axis=1)
    _data_frame.drop('TotalWins', inplace=True, axis=1)
    _data_frame.drop('TotalDeaths', inplace=True, axis=1)
    _data_frame.drop('NpcsInteracted', inplace=True, axis=1)
    _data_frame.drop('RoomsEntered', inplace=True, axis=1)
    _data_frame.drop('TotalRooms', inplace=True, axis=1)
    _data_frame.drop('UniqueRoomsEntered', inplace=True, axis=1)
    _data_frame.drop('TotalTreasure', inplace=True, axis=1)
    return _data_frame


def split_preform_answers(_data_frame):
    _data_frame[['A', 'C', 'I', 'M']] = _data_frame['PreFormAnswers'].str.split(',', expand=True)
    _data_frame['A'] = _data_frame['A'].apply(lambda x: ''.join((char for char in x if char.isdigit())))
    _data_frame['C'] = _data_frame['C'].apply(lambda x: ''.join((char for char in x if char.isdigit())))
    _data_frame['I'] = _data_frame['I'].apply(lambda x: ''.join((char for char in x if char.isdigit())))
    _data_frame['M'] = _data_frame['M'].apply(lambda x: ''.join((char for char in x if char.isdigit())))
    return _data_frame

# -----------------------------
# DATA AUGMENTATION
# -----------------------------

def augment_dataframe_with_scaling(df, y, n, scale_factor):
    """
    Augments the DataFrame by adding `n` new rows, scaling numeric columns by a given factor.
    The new rows inherit the same 'y' values as their corresponding sampled rows.

    Parameters:
        df (pd.DataFrame): Original DataFrame with numeric columns and a 'y' column.
        n (int): Number of new rows to generate.
        scale_factor (float): The factor by which numeric columns will be scaled.

    Returns:
        pd.DataFrame: DataFrame with the original and new rows.
    """
    # Identify numeric columns, excluding 'y'
    numeric_columns = df.select_dtypes(include=['number']).columns.difference(y)
    # Randomly sample `n` rows from the original DataFrame
    sampled_rows = df.sample(n=n, replace=True).reset_index(drop=True)

    # Scale numeric columns
    for col in numeric_columns:
        sampled_rows[col] *= scale_factor

    # Append the scaled rows to the original DataFrame
    augmented_df = pd.concat([df, sampled_rows], ignore_index=True)

    return augmented_df


def augment_data_random(_df, y, n):
    """
    Augments the DataFrame by adding `n` new rows.
    - Adds noise to numerical columns using a normal distribution.
    - Retains the same 'y' values for the newly added rows as their corresponding sampled rows.

    Parameters:
        _df (pd.DataFrame): Original DataFrame with numeric columns and a 'y' column.
        y (list): the column(s) that will have their value retained
        n (int): Number of new rows to generate.

    Returns:
        pd.DataFrame: DataFrame with the original and new rows.
    """
    # Identify float and integer columns, excluding 'y'
    float_columns = _df.select_dtypes(include=['float']).columns.difference(y)
    int_columns = _df.select_dtypes(include=['integer']).columns.difference(y)

    # Randomly sample `n` rows from the original DataFrame
    sampled_rows = _df.sample(n=n, replace=True).reset_index(drop=True)

    # Add noise to float columns
    for col in float_columns:
        if random.randint(1, 100) > 50:
            std = _df[col].std()
            noise = np.random.normal(loc=0, scale=std, size=n)
            sampled_rows[col] += noise

    for col in int_columns:
        if random.randint(1, 100) > 50:
            std = _df[col].std()
            noise = np.random.normal(loc=0, scale=std, size=n)
            sampled_rows[col] += noise

    # Ensure integer columns remain integers
    sampled_rows[int_columns] = sampled_rows[int_columns].round().astype(int)

    # Append the sampled (augmented) rows to the original DataFrame
    augmented_df = pd.concat([_df, sampled_rows], ignore_index=True)
    return augmented_df

def leave_one_out_split(df, user_col='PlayerId'):
    train_list, test_list = [], []
    for u, grp in df.groupby(user_col):
        if len(grp)<2:
            train_list.append(grp); continue
        test_list.append(grp.sample(1, random_state=42))
        train_list.append(grp.drop(test_list[-1].index))
    return pd.concat(train_list), pd.concat(test_list)

In [71]:
# -----------------------------
# ANÁLISES
# -----------------------------
def inspect_dataset(user_frame, dungeon_frame):
    profiles = user_frame['PreFormAnswers'].apply(tuple)

    print("Combinações únicas:")
    print(sorted(profiles.unique()))

    print("\nQuantidade de combinações únicas:")
    print(profiles.nunique())

    print("\nFrequência de cada combinação:")
    print(profiles.value_counts())

    dungeon_frame = pd.merge(
        dungeon_frame,
        user_frame[['PlayerId', 'PreFormAnswers']],
        on=['PlayerId'],
        how='left'
    )

    print("\nFrequência no dungeon_frame final:")
    print(dungeon_frame['PreFormAnswers'].value_counts())

    # Listar todas as colunas
    colunas = dungeon_frame.columns.tolist()

    print(f"\nTotal de colunas: {len(colunas)}\n")

    # Exibir em ordem alfabética numerada
    for i, col in enumerate(sorted(colunas), 1):
        print(f"{i:3d}. {col}")

def analyze_user_item_matrix(user_item_matrix):

    print("------------- VERIFICAÇÃO DOS DADOS -------------")
    print("Número de usuários:", user_item_matrix.shape[0])
    print("Número de itens:", user_item_matrix.shape[1])

    print("\nQuantidade média de avaliações por usuário:")
    print(user_item_matrix.count(axis=1).mean())

    print("\nQuantidade média de usuários por item:")
    print(user_item_matrix.count(axis=0).mean())

    print("\n------------- VARIÂNCIA ITEM POR ITEM -------------")
    print("Variância de cada item:")
    print(user_item_matrix.var())

In [72]:
# -----------------------------
# RECOMENDAÇÃO
# -----------------------------
def recommend_from_other_user(target_user, other_user, user_item_matrix):

    print(f"\nRecomendando para usuário {target_user} com base no usuário {other_user}")

    # Itens que target_user já jogou
    target_items = user_item_matrix.loc[target_user]

    # Itens que other_user já jogou
    other_items = user_item_matrix.loc[other_user]

    # Pegar apenas itens que:
    # - other_user jogou (rating > 0)
    # - target_user NÃO jogou (rating == 0)
    candidate_items = other_items[
        (other_items > 0) & (target_items == 0)
    ]

    if len(candidate_items) == 0:
        print("Nenhum item novo para recomendar.")
        return

    # Ordenar pelos melhores ratings do other_user
    recommendations = candidate_items.sort_values(ascending=False)

    print("\nItens recomendados:")
    print(recommendations.head(5))


# ---------------- RECOMENDAÇÃO USER-BASED EXPLICÁVEL ---------------- #

def recommend_user_based_explain(target_user, user_item_matrix, user_similarity_df, top_n=5):

    print(f"\nRecomendações para usuário {target_user}")

    # Similaridade do usuário alvo com todos os outros
    similarities = user_similarity_df.loc[target_user].drop(target_user)
    # Pegar itens que o usuário ainda NÃO jogou
    user_ratings = user_item_matrix.loc[target_user]
    items_not_played = user_ratings[user_ratings == 0].index

    predictions = {}
    explanations = {}

    for item in items_not_played:
        # Usuários que avaliaram esse item
        users_who_rated = user_item_matrix[user_item_matrix[item] > 0].index

        numerator = 0
        denominator = 0
        contributors = []

        for other_user in users_who_rated:

            sim = similarities.get(other_user, 0)

            if sim > 0:# só similaridade positiva
                rating = user_item_matrix.loc[other_user, item]
                contribution = sim * rating

                numerator += contribution
                denominator += abs(sim)

                contributors.append({
                    "user": other_user,
                    "similarity": sim,
                    "rating": rating,
                    "contribution": contribution
                })

        if denominator > 0:
            predicted_rating = numerator / denominator
            predictions[item] = predicted_rating
            explanations[item] = contributors

    if len(predictions) == 0:
        print("Nenhum item possível para recomendar.")
        return

    recommendations = (
        pd.Series(predictions)
        .sort_values(ascending=False)
        .head(top_n)
    )

    '''print("\nItens recomendados:")
    print(recommendations)'''

    # Mostrar explicação detalhada
    for item in recommendations.index:
        print("\n---------------------------------------")
        print(f"Explicação para {item}")
        print("---------------------------------------")

        contribs = sorted(
            explanations[item],
            key=lambda x: x["contribution"],
            reverse=True
        )

        for c in contribs[:5]:  # top 5 contribuidores
            print(
                f"Usuário {c['user']} | "
                f"similaridade={round(c['similarity'],3)} | "
                f"nota={round(c['rating'],2)} | "
                f"contribuição={round(c['contribution'],3)}"
            )
# Recommend items similar to 'Item1'
def recommend_similar_items(item, similarity_df, top_n=3):
  return similarity_df[item].sort_values(ascending=False)[1:top_n+1]

def print_user_ratings(user_item_matrix, user_similarity_df, user1_idx=0, user2_idx=1):
    # seleciona usuários
    u1 = user_similarity_df.index[user1_idx]
    u2 = user_similarity_df.index[user2_idx]

    print("Notas do usuário 1:")
    print(user_item_matrix.loc[u1].dropna())

    print("\nNotas do usuário 2:")
    print(user_item_matrix.loc[u2].dropna())

def multi_view_recommendation(df, target_user):

    quest_df  = build_quest_df(df)
    enemy_df  = build_enemy_df(df)
    struct_df = build_struct_df(df)

    # gerar matrizes
    def build_matrix(d):
        return d.pivot_table(index='User', columns='Item', values='Rating', fill_value=0)

    quest_matrix  = build_matrix(quest_df)
    enemy_matrix  = build_matrix(enemy_df)
    struct_matrix = build_matrix(struct_df)

    # similaridade
    def sim(m):
        return pd.DataFrame(
            cosine_similarity(m),
            index=m.index,
            columns=m.index
        )

    quest_sim  = sim(quest_matrix)
    enemy_sim  = sim(enemy_matrix)
    struct_sim = sim(struct_matrix)

    print("\n===== QUEST =====")
    recommend_user_based_explain(target_user, quest_matrix, quest_sim)

    print("\n===== ENEMY =====")
    recommend_user_based_explain(target_user, enemy_matrix, enemy_sim)

    print("\n===== STRUCT =====")
    recommend_user_based_explain(target_user, struct_matrix, struct_sim)

def recommend_hybrid(user,
                     user_item_matrix,
                     item_similarity_df,
                     user_similarity_df,
                     top_n=5,
                     alpha=0.5):

    items = user_item_matrix.columns
    predictions = {}

    for item in items:
        if user_item_matrix.loc[user, item] == 0:  # não avaliou
            predictions[item] = predict_hybrid(
                user,
                item,
                user_item_matrix,
                item_similarity_df,
                user_similarity_df,
                alpha
            )

    ranked = sorted(predictions.items(),
                    key=lambda x: x[1],
                    reverse=True)

    return ranked[:top_n]

In [73]:
# -----------------------------
# CÁLCULOS
# -----------------------------
def evaluate_rating_metrics(predictions_df, test_matrix):

    y_true = []
    y_pred = []

    for _, row in predictions_df.iterrows():
        user = row["User"]
        item = row["Item"]

        if user in test_matrix.index and item in test_matrix.columns:
            true_rating = test_matrix.loc[user, item]
            if true_rating != 0:
                y_true.append(true_rating)
                y_pred.append(row["Prediction"])

    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)

    return rmse, mae

def compute_rmse(test_df, prediction_col, rating_col):
    """
    Calcula RMSE ignorando:
        - valores NaN
        - casos sem predição
        - divisão por zero
    """

    # Manter apenas linhas com valor real e predito
    mask = (
        test_df[rating_col].notna() &
        test_df[prediction_col].notna()
    )

    y_true = test_df.loc[mask, rating_col].values
    y_pred = test_df.loc[mask, prediction_col].values

    if len(y_true) == 0:
        raise ValueError("Nenhuma predição válida para calcular RMSE.")

    rmse = np.sqrt(np.mean((y_true - y_pred) ** 2))

    return rmse

def precision_at_k(rec_df, test_df, k=5, threshold=1.0):
    precision_list = []

    for user in test_df["User"].unique():

        top_k = rec_df[rec_df["User"] == user].head(k)["Item"].values
        true_items = test_df[(test_df["User"] == user) & (test_df["Rating"] >= threshold)]["Item"].values

        if len(true_items) == 0:
            continue

        hits = len(set(top_k) & set(true_items))
        precision_list.append(hits / k)

    return np.mean(precision_list)
def recall_at_k(rec_df, test_df, k=5, threshold=1.0):
    recall_list = []

    for user in test_df["User"].unique():

        top_k = rec_df[rec_df["User"] == user].head(k)["Item"].values
        true_items = test_df[(test_df["User"] == user) & (test_df["Rating"] >= threshold)]["Item"].values

        if len(true_items) == 0:
            continue

        hits = len(set(top_k) & set(true_items))
        recall_list.append(hits / len(true_items))

    return np.mean(recall_list)
def hit_rate_at_k(rec_df, test_df, k=5):
    hits = 0
    users = test_df["User"].unique()

    for user in users:
        top_k = rec_df[rec_df["User"] == user].head(k)["Item"].values
        true_items = test_df[test_df["User"] == user]["Item"].values

        if len(set(top_k) & set(true_items)) > 0:
            hits += 1

    return hits / len(users)
def mrr(rec_df, test_df):
    rr_total = []

    for user in test_df["User"].unique():

        top_k = rec_df[rec_df["User"] == user].reset_index()

        true_items = set(test_df[test_df["User"] == user]["Item"])

        rank = 0

        for i, row in top_k.iterrows():
            if row["Item"] in true_items:
                rank = i + 1
                break

        if rank > 0:
            rr_total.append(1 / rank)

    return np.mean(rr_total)

def similiarity_pearson(user_item_matrix):
    # Pearson item-item
    item_similarity_pearson = user_item_matrix.corr(method='pearson')

    print("\nITEM POR ITEM (PEARSON):\n")
    print(item_similarity_pearson)
    item_similarity_df = item_similarity_pearson

    user_similarity_pearson = user_item_matrix.T.corr(method='pearson')
    user_similarity_pearson = user_similarity_pearson.fillna(0)

    print("\nUSUARIO POR USUARIO (PEARSON):\n")
    print(user_similarity_pearson)

    user_similarity_df  = user_similarity_pearson

    return item_similarity_df, user_similarity_df

def similiarity_cosine(user_item_matrix):
    # Compute item similarity using cosine similarity
    item_similarity = cosine_similarity(user_item_matrix.T)
    item_similarity_df = pd.DataFrame(item_similarity, index=user_item_matrix.columns, columns=user_item_matrix.columns)

    #centralizando
    user_item_centered = user_item_matrix.sub(user_item_matrix.mean(axis=1), axis=0)
    #print("\nconsseno cetnralizado:\n")
    #print(user_item_centered)

    # Similaridade entre itens
    # item por item
    item_similarity_centered = cosine_similarity(user_item_centered.T)
    item_similarity_centered_df = pd.DataFrame(item_similarity_centered, index=user_item_centered.columns, columns=user_item_centered.columns)

    #print("\nconsseno:\n")
    #print(item_similarity_df)

    #print("\nconsseno cetnralizado:\n")
    #print(item_similarity_centered)

    # usuario por usuário
    user_similarity = cosine_similarity(user_item_centered)
    user_similarity_df = pd.DataFrame( user_similarity, index=user_item_matrix.index, columns=user_item_matrix.index )
    #print("\nUSUARIO POR USUARIO:\n")
    #print(user_similarity_df.iloc[:50, :50])

    return item_similarity_df, user_similarity_df

def compute_performance_rating(df):
    df['TotalLostHealth_norm'] = minmax(df['TotalLostHealth'])

    return (
        df['SuccessRate']
        - df['TotalLostHealth_norm']
        + df['EnemyKillRate']
        + df['MapCompletion']
    )

def build_item_from_structure(df):
    df['TotalRooms'] = df['TotalRooms'].replace(0, 1)

    return (
        "R" + df['TotalRooms'].astype(str) +
        "_K" + df['TotalKeys'].astype(str))# +        "_RR" + (df['RoomsEntered'] - df['UniqueRoomsEntered']).astype(str)    )

def compute_hybrid_rating(df, alpha=0.5):
    df_valid = df['PostFormAnswers'].apply(mean_post_forms)
    df_comp = compute_performance_rating(df)

    return alpha * df_comp + (1 - alpha) * df_valid

def mean_post_forms(x):
    return sum(x)/len(x) if isinstance(x, list) else x

def minmax(series):
    return (series - series.min()) / (series.max() - series.min() + 1e-9)

In [74]:
# -----------------------------
# ESTRUTURA
# -----------------------------

def get_bucket_value(attr, bucket):
    """
    Converte o bucket (ex: 'R1', 'K2') para um valor legível
    """
    if attr == 'TotalRooms':
        mapa = {
            'R1': '10-15 salas',
            'R2': '15-20 salas',
            'R3': '20-25 salas',
            'R4': '25-30 salas',
            'R0': 'dados insuficientes'
        }
    elif attr == 'TotalKeys':
        mapa = {
            'K1': '0-2 chaves',
            'K2': '2-4 chaves',
            'K3': '4-6 chaves',
            'K0': 'dados insuficientes'
        }
    elif attr == 'TotalLocks':
        mapa = {
            'L1': '0-2 fechaduras',
            'L2': '2-4 fechaduras',
            'L3': '4-6 fechaduras',
            'L0': 'dados insuficientes'
        }
    elif attr == 'TotalEnemies':
        mapa = {
            'E1': '0-10 inimigos',
            'E2': '10-20 inimigos',
            'E3': '20-30 inimigos',
            'E4': '30-50 inimigos',
            'E5': '50+ inimigos'
        }
    elif attr == 'TotalNpcs':
        mapa = {
            'N1': '0-2 NPCs',
            'N2': '2-4 NPCs',
            'N3': '4-6 NPCs',
            'N4': '6-10 NPCs',
            'N0': 'dados insuficientes'
        }
    else:
        return bucket

    return mapa.get(bucket, bucket)

def build_collab_df_granular(_df, method, attribute=None):
    """
    Se attribute = None: item = LevelName, rating = PostFormAnswers
    Se attribute = 'TotalLocks': item = granular, rating = PostFormAnswers
    """
    df = _df.copy()

    if attribute is None:
        # Abordagem tradicional
        item_col = 'LevelName'
    else:
        # Abordagem granular: combinar atributos
        # Garantir que não há NaN e converter para número
        df['TotalRooms'] = pd.to_numeric(df['TotalRooms'], errors='coerce').fillna(0)
        df['TotalKeys'] = pd.to_numeric(df['TotalKeys'], errors='coerce').fillna(0)
        df['TotalLocks'] = pd.to_numeric(df['TotalLocks'], errors='coerce').fillna(0)

        # Discretizar com .astype(str) para evitar erro de concatenação
        df['RoomBucket'] = pd.cut(df['TotalRooms'], bins=[0, 10, 20, 30, 50], labels=['R1','R2','R3','R4']).astype(str)
        df['KeyBucket'] = pd.cut(df['TotalKeys'], bins=[0, 3, 5, 10], labels=['K1','K2','K3']).astype(str)
        df['LockBucket'] = pd.cut(df['TotalLocks'], bins=[0, 3, 5, 10], labels=['L1','L2','L3']).astype(str)

        # Substituir NaN por string vazia
        df['RoomBucket'] = df['RoomBucket'].fillna('R0')
        df['KeyBucket'] = df['KeyBucket'].fillna('K0')
        df['LockBucket'] = df['LockBucket'].fillna('L0')

        df['Item'] = df['RoomBucket'] + "_" + df['KeyBucket'] + "_" + df['LockBucket']
        item_col = 'Item'

    # Rating SEMPRE é o desempenho do jogador
    rating_col = 'PostFormAnswers'

    # Garantir que as colunas existem
    if item_col not in df.columns:
        print(f"⚠️ Coluna {item_col} não encontrada!")
        return pd.DataFrame(columns=['User', 'Item', 'Rating'])

    result = df[['PlayerId', item_col, rating_col]].copy()
    result.columns = ['User', 'Item', 'Rating']

    # Remover linhas com Rating NaN
    result = result.dropna(subset=['Rating'])

    # Calcular média do PostFormAnswers
    result['Rating'] = result['Rating'].apply(lambda x: sum(x)/len(x) if isinstance(x, list) else x)
    result['Rating'] = minmax(result['Rating'])

    return result[['User', 'Item', 'Rating']]

def build_collab_df(_df, method, item_col_pass='PostFormAnswers'):
    # ----------------------------
    # ITEM PROFILE
    # ----------------------------
    if method in ITEM_PROFILE_METHODS:

        if method in {1,2}:
            item_col = 'PreFormAnswers'
        else:
            item_col = 'LevelName'

        cols = ['PlayerId', item_col, 'PostFormAnswers']

        df = _df[cols].copy()
        df.columns = ['User', 'Item', 'Rating']
        df['Rating'] = df['Rating'].apply(mean_post_forms)
        df['Rating'] = minmax(df['Rating'])
        return df[['User','Item','Rating']]
    # ----------------------------
    # USER PROFILE (performance)
    # ----------------------------
    elif method in USER_PROFILE_METHODS:

        if method in {3,4}:
            item_col = 'PreFormAnswers'
        else:
            item_col = 'LevelName'

        cols = ['PlayerId', item_col, 'Success Rate',
                'TotalLostHealth', 'EnemyKillRate', 'MapCompletion']

        df = _df[cols].copy()
        df.columns = ['User', 'Item', 'SuccessRate',
                      'TotalLostHealth', 'EnemyKillRate', 'MapCompletion']


        df['Rating'] = compute_performance_rating(df)
        df = df[['User','Item','Rating']]
        df['Rating'] = minmax(df['Rating'])
        return df[['User','Item','Rating']]

    # ----------------------------
    # STRUCTURE BASED
    # ----------------------------
    elif method in STRUCTURE_ONLY_METHODS.union(STRUCTURE_PERF_METHODS):

        cols = ['PlayerId','TotalRooms','TotalKeys',]
                #'RoomsEntered','UniqueRoomsEntered']

        if method in STRUCTURE_ONLY_METHODS:
            cols.append('PostFormAnswers')
        else:
            cols += ['Success Rate','TotalLostHealth',
                     'EnemyKillRate','MapCompletion']

        df = _df[cols].copy()
        if method in STRUCTURE_ONLY_METHODS:
          df.columns = ['User', 'TotalRooms', 'TotalKeys', 'Rating']# 'RoomsEntered', 'UniqueRoomsEntered', 'Rating']
        else:
            df.columns = ['User', 'TotalRooms', 'TotalKeys', 'SuccessRate', 'TotalLostHealth', 'EnemyKillRate', 'MapCompletion']#'RoomsEntered', 'UniqueRoomsEntered', 'SuccessRate', 'TotalLostHealth', 'EnemyKillRate', 'MapCompletion']

        df['Item'] = build_item_from_structure(df)

        if method in STRUCTURE_ONLY_METHODS:
            df['Rating'] = df['Rating'].apply(mean_post_forms)
        else:

            df['Rating'] = compute_performance_rating(df)

        df['Rating'] = minmax(df['Rating'])
        df = df[['User','Item','Rating']]
        return df[['User','Item','Rating']]

    # --------------------------------------------------
    # HYBRID PREFORM (14,15)
    # --------------------------------------------------
    elif method in HYBRID_PREFORM_METHODS:

        cols = ['PlayerId','PreFormAnswers','PostFormAnswers',
                'Success Rate','TotalLostHealth',
                'EnemyKillRate','MapCompletion']

        df = _df[cols].copy()
        df.columns = ['User','Item','PostFormAnswers',
                      'SuccessRate','TotalLostHealth',
                      'EnemyKillRate','MapCompletion']

        df['Rating'] = compute_hybrid_rating(df)
        df['Rating'] = minmax(df['Rating'])
        return df[['User','Item','Rating']]


    # --------------------------------------------------
    # HYBRID LEVEL (16,17)
    # --------------------------------------------------
    elif method in HYBRID_LEVEL_METHODS:

        cols = ['PlayerId','LevelName','PostFormAnswers',
                'Success Rate','TotalLostHealth',
                'EnemyKillRate','MapCompletion']

        df = _df[cols].copy()
        df.columns = ['User','Item','PostFormAnswers',
                      'SuccessRate','TotalLostHealth',
                      'EnemyKillRate','MapCompletion']

        df['Rating'] = compute_hybrid_rating(df)
        df['Rating'] = minmax(df['Rating'])
        return df[['User','Item','Rating']]


    # --------------------------------------------------
    # HYBRID STRUCTURE (18,19)
    # --------------------------------------------------
    elif method in HYBRID_STRUCTURE_METHODS:

        cols = ['PlayerId','TotalRooms','TotalKeys',
                'RoomsEntered','UniqueRoomsEntered',
                'PostFormAnswers','Success Rate',
                'TotalLostHealth','EnemyKillRate','MapCompletion']

        df = _df[cols].copy()
        df.columns = ['User','TotalRooms','TotalKeys',
                      'RoomsEntered','UniqueRoomsEntered',
                      'PostFormAnswers','SuccessRate',
                      'TotalLostHealth','EnemyKillRate','MapCompletion']

        df['Item'] = build_item_from_structure(df)
        df['Rating'] = compute_hybrid_rating(df)
        df['Rating'] = minmax(df['Rating'])
        return df[['User','Item','Rating']]
      ###################################################
    elif method in UNNAMED:
      print(f"\n🎯 MÉTODO {method} - Atributo sendo processado: {item_col_pass}")

      # Item SEMPRE é LevelName
      item_col = 'LevelName'

      # Rating é o ATRIBUTO (TotalRooms, TotalKeys, etc.)
      rating_col = item_col_pass

      cols = ['PlayerId', item_col, rating_col]
      df = _df[cols].copy()
      df.columns = ['User', 'Item', 'Rating']

      # 🔥 NÃO aplicar mean_post_forms! O atributo já é numérico
      # Apenas normalizar e discretizar
      df['Rating'] = minmax(df['Rating'])

      df['Rating'] = pd.cut(df['Rating'], bins=3, labels=[0.33, 0.66, 1.0])
      df['Rating'] = df['Rating'].astype(float)

      return df[['User','Item','Rating']]

    else:
        raise ValueError(f"Método {method} inválido.")

    return df

def train_test_split_interactions(df, test_size=0.2):
    train, test = train_test_split(df, test_size=test_size, random_state=42)
    return train, test

In [75]:
# -----------------------------
# PREDICT
# -----------------------------
def predict_item_based(user, item, user_item_matrix, item_similarity_df, k=20):
    if item not in item_similarity_df.columns or user not in user_item_matrix.index:
        return 0  # ou np.nan
    similar_items = item_similarity_df[item]
    user_ratings = user_item_matrix.loc[user]
    # 👉 Filtrar itens não avaliados pelo usuário
    mask = user_ratings != 0
    similar_items = similar_items[mask]; user_ratings = user_ratings[mask]
    if similar_items.empty:
        return 0
    # (Opcional) usar apenas os top-K vizinhos
    similar_items = similar_items.sort_values(ascending=False).head(k)
    user_ratings = user_ratings.loc[similar_items.index]
    numerator = (similar_items * user_ratings).sum()
    denominator = similar_items.abs().sum()
    return numerator/denominator if denominator != 0 else 0

def predict_user_based(user, item, user_item_matrix, user_similarity_df, k=20):
    if user not in user_similarity_df.index or item not in user_item_matrix.columns:
        return 0
    similar_users = user_similarity_df.loc[user].drop(user)
    item_ratings = user_item_matrix[item]
    mask = item_ratings != 0
    similar_users = similar_users[mask]; item_ratings = item_ratings[mask]
    if similar_users.empty:
        return 0
    similar_users = similar_users.sort_values(ascending=False).head(k)
    item_ratings = item_ratings.loc[similar_users.index]
    numerator = (similar_users * item_ratings).sum()
    denominator = similar_users.abs().sum()
    return numerator/denominator if denominator != 0 else 0


def predict_hybrid(user, item,
                   user_item_matrix,
                   item_similarity_df,
                   user_similarity_df,
                   alpha=0.5):

    score_item = predict_item_based(
        user, item,
        user_item_matrix,
        item_similarity_df
    )

    score_user = predict_user_based(
        user, item,
        user_item_matrix,
        user_similarity_df
    )

    return alpha * score_item + (1 - alpha) * score_user


def predict_all_users(user_item_matrix, similarity_matrix, method="user"):
    predictions = []

    for user in user_item_matrix.index:
        for item in user_item_matrix.columns:
            if user_item_matrix.loc[user, item] == 0:

                if method == "user":
                    pred = predict_user_based(user, item, user_item_matrix, similarity_matrix)
                else:
                    pred = predict_item_based(user, item, user_item_matrix, similarity_matrix)

                predictions.append((user, item, pred))

    return pd.DataFrame(predictions, columns=["User", "Item", "Prediction"])

In [76]:
# -----------------------------
# RECOMENDAR POR COLUNA
# -----------------------------
def aggregate_recommendations_by_enemy_difficulty(recommended_levels_df, dungeon_metadata_df, user_profile_df, ratings_df, top_n=5):
    """
    Analisa a relacao entre rating e dificuldade para inferir preferencia
    """

    enemy_columns = [
        'TotalEnemies',
        'EnemyKillRate',
        'TotalLostHealth',
        'MaxCombo',
        'TotalDeaths'
    ]

    available_cols = [col for col in enemy_columns if col in dungeon_metadata_df.columns]

    if len(available_cols) == 0:
        print("Nenhuma coluna de inimigos encontrada")
        return pd.DataFrame()

    resultado = []

    for user in recommended_levels_df['User'].unique():
        user_recs = recommended_levels_df[recommended_levels_df['User'] == user].head(top_n).copy()

        user_recs = user_recs.merge(dungeon_metadata_df[['LevelName'] + available_cols], left_on='Item', right_on='LevelName', how='left')

        user_recs = user_recs.merge(ratings_df[ratings_df['User'] == user][['Item', 'Rating']], on='Item', how='left')

        if len(user_recs) == 0:
            continue

        user_profile = user_profile_df[user_profile_df['PlayerId'] == user]
        preform = user_profile['PreFormAnswers'].iloc[0] if len(user_profile) > 0 else 'N/A'

        rating_mean = user_recs['Rating'].mean()
        kill_rate_mean = user_recs['EnemyKillRate'].mean() if 'EnemyKillRate' in available_cols else 0

        if rating_mean >= 0.7:
            if kill_rate_mean >= 0.7:
                recomendacao = "recomendar dungeons com MUITOS inimigos (kill_rate alto)"
            elif kill_rate_mean >= 0.5:
                recomendacao = "recomendar dungeons com quantidade MEDIA de inimigos"
            else:
                recomendacao = "recomendar dungeons com POUCOS inimigos"
        elif rating_mean >= 0.4:
            if kill_rate_mean >= 0.7:
                recomendacao = "recomendar dungeons com quantidade MEDIA de inimigos"
            elif kill_rate_mean >= 0.5:
                recomendacao = "recomendar dungeons com POUCOS inimigos"
            else:
                recomendacao = "recomendar dungeons com MUITO POUCOS inimigos"
        else:
            recomendacao = "recomendar dungeons com MUITO POUCOS inimigos (usuario esta tendo dificuldade)"

        resultado.append({
            'User': user,
            'PreFormAnswers': preform,
            'rating_medio': round(rating_mean, 2),
            'kill_rate_medio': round(kill_rate_mean, 2),
            'recomendacao_inimigos': recomendacao
        })

    return pd.DataFrame(resultado)

def aggregate_recommendations_by_npcs(recommended_levels_df, dungeon_metadata_df, user_profile_df, ratings_df, top_n=5):
    """
    Analisa preferencia por NPCs baseado no rating
    """

    npc_columns = ['TotalNpcs', 'NpcsInteracted']
    available_cols = [col for col in npc_columns if col in dungeon_metadata_df.columns]

    if len(available_cols) == 0:
        print("Nenhuma coluna de NPCs encontrada")
        return pd.DataFrame()

    resultado = []

    for user in recommended_levels_df['User'].unique():
        user_recs = recommended_levels_df[recommended_levels_df['User'] == user].head(top_n).copy()
        user_recs = user_recs.merge(dungeon_metadata_df[['LevelName'] + available_cols], left_on='Item', right_on='LevelName', how='left')
        user_recs = user_recs.merge(ratings_df[ratings_df['User'] == user][['Item', 'Rating']], on='Item', how='left')

        if len(user_recs) == 0:
            continue

        user_profile = user_profile_df[user_profile_df['PlayerId'] == user]
        preform = user_profile['PreFormAnswers'].iloc[0] if len(user_profile) > 0 else 'N/A'

        rating_mean = user_recs['Rating'].mean()
        npcs_total_mean = user_recs['TotalNpcs'].mean() if 'TotalNpcs' in available_cols else 0
        npcs_interacted_mean = user_recs['NpcsInteracted'].mean() if 'NpcsInteracted' in available_cols else 0

        interacao_ratio = npcs_interacted_mean / npcs_total_mean if npcs_total_mean > 0 else 0

        if rating_mean >= 0.7:
            if interacao_ratio >= 0.7:
                recomendacao = "recomendar dungeons com MUITOS NPCs para interagir"
            elif interacao_ratio >= 0.4:
                recomendacao = "recomendar dungeons com quantidade MEDIA de NPCs"
            else:
                recomendacao = "recomendar dungeons com POUCOS NPCs"
        elif rating_mean >= 0.4:
            if interacao_ratio >= 0.7:
                recomendacao = "recomendar dungeons com quantidade MEDIA de NPCs"
            else:
                recomendacao = "recomendar dungeons com POUCOS NPCs"
        else:
            recomendacao = "recomendar dungeons com MUITO POUCOS NPCs"

        resultado.append({
            'User': user,
            'PreFormAnswers': preform,
            'rating_medio': round(rating_mean, 2),
            'interacao_ratio': round(interacao_ratio, 2),
            'recomendacao_npcs': recomendacao
        })

    return pd.DataFrame(resultado)

def aggregate_recommendations_by_progress(recommended_levels_df, dungeon_metadata_df, user_profile_df, ratings_df, top_n=5):
    """
    Analisa preferencia por progresso (tesouros e tempo)
    """

    progress_columns = ['TotalTreasure', 'TreasuresCollected', 'TimeToFinish']
    available_cols = [col for col in progress_columns if col in dungeon_metadata_df.columns]

    if len(available_cols) == 0:
        print("Nenhuma coluna de progresso encontrada")
        return pd.DataFrame()

    resultado = []

    for user in recommended_levels_df['User'].unique():
        user_recs = recommended_levels_df[recommended_levels_df['User'] == user].head(top_n).copy()
        user_recs = user_recs.merge(dungeon_metadata_df[['LevelName'] + available_cols], left_on='Item', right_on='LevelName', how='left')
        user_recs = user_recs.merge(ratings_df[ratings_df['User'] == user][['Item', 'Rating']], on='Item', how='left')

        if len(user_recs) == 0:
            continue

        user_profile = user_profile_df[user_profile_df['PlayerId'] == user]
        preform = user_profile['PreFormAnswers'].iloc[0] if len(user_profile) > 0 else 'N/A'

        rating_mean = user_recs['Rating'].mean()
        treasure_ratio = (user_recs['TreasuresCollected'] / user_recs['TotalTreasure']).mean() if 'TotalTreasure' in available_cols and 'TreasuresCollected' in available_cols else 0
        time_mean = user_recs['TimeToFinish'].mean() if 'TimeToFinish' in available_cols else 0

        if rating_mean >= 0.7:
            if treasure_ratio >= 0.7:
                recomendacao = f"recomendar dungeons com MUITOS tesouros (ratio {treasure_ratio:.2f})"
            elif treasure_ratio >= 0.4:
                recomendacao = f"recomendar dungeons com quantidade MEDIA de tesouros"
            else:
                recomendacao = f"recomendar dungeons com POUCOS tesouros"
        else:
            recomendacao = f"recomendar dungeons com POUCOS tesouros"

        resultado.append({
            'User': user,
            'PreFormAnswers': preform,
            'rating_medio': round(rating_mean, 2),
            'treasure_ratio': round(treasure_ratio, 2),
            'tempo_medio': round(time_mean, 2),
            'recomendacao_progresso': recomendacao
        })

    return pd.DataFrame(resultado)

def aggregate_recommendations_by_structure(recommended_levels_df, dungeon_metadata_df, user_profile_df, ratings_df, top_n=5):
    """
    Analisa preferencia por estrutura (salas, chaves, fechaduras)
    """

    structure_columns = ['TotalRooms', 'TotalKeys', 'TotalLocks', 'UniqueRoomsEntered', 'RoomsEntered']
    available_cols = [col for col in structure_columns if col in dungeon_metadata_df.columns]

    if len(available_cols) == 0:
        print("Nenhuma coluna estrutural encontrada")
        return pd.DataFrame()

    resultado = []

    for user in recommended_levels_df['User'].unique():
        user_recs = recommended_levels_df[recommended_levels_df['User'] == user].head(top_n).copy()
        user_recs = user_recs.merge(dungeon_metadata_df[['LevelName'] + available_cols], left_on='Item', right_on='LevelName', how='left')
        user_recs = user_recs.merge(ratings_df[ratings_df['User'] == user][['Item', 'Rating']], on='Item', how='left')

        if len(user_recs) == 0:
            continue

        user_profile = user_profile_df[user_profile_df['PlayerId'] == user]
        preform = user_profile['PreFormAnswers'].iloc[0] if len(user_profile) > 0 else 'N/A'

        rating_mean = user_recs['Rating'].mean()
        rooms_mean = user_recs['TotalRooms'].mean() if 'TotalRooms' in available_cols else 0
        keys_mean = user_recs['TotalKeys'].mean() if 'TotalKeys' in available_cols else 0
        locks_mean = user_recs['TotalLocks'].mean() if 'TotalLocks' in available_cols else 0

        revisit_ratio = (user_recs['RoomsEntered'] / user_recs['UniqueRoomsEntered']).mean() if 'RoomsEntered' in available_cols and 'UniqueRoomsEntered' in available_cols else 0

        if rating_mean >= 0.7:
            if rooms_mean >= 20:
                recomendacao = "recomendar dungeons GRANDES (muitas salas)"
            elif rooms_mean >= 10:
                recomendacao = "recomendar dungeons MEDIAS"
            else:
                recomendacao = "recomendar dungeons PEQUENAS"
        else:
            recomendacao = "recomendar dungeons PEQUENAS (usuario prefere dungeons curtas)"

        resultado.append({
            'User': user,
            'PreFormAnswers': preform,
            'rating_medio': round(rating_mean, 2),
            'rooms_medio': round(rooms_mean, 1),
            'keys_medio': round(keys_mean, 1),
            'locks_medio': round(locks_mean, 1),
            'revisit_ratio': round(revisit_ratio, 2),
            'recomendacao_estrutura': recomendacao
        })

    return pd.DataFrame(resultado)

def aggregate_recommendations_by_quest(recommended_levels_df, dungeon_metadata_df, user_profile_df, ratings_df, top_n=5):
    """
    recommended_levels_df: DataFrame com colunas ['User', 'Item', 'Score'] onde Item = LevelName
    dungeon_metadata_df: DataFrame com colunas ['LevelName', 'CompletedCreativityQuests', ...]
    user_profile_df: DataFrame com colunas ['PlayerId', 'PreFormAnswers']
    ratings_df: DataFrame com colunas ['User', 'Item', 'Rating']
    """

    quest_columns = [
        'CompletedCreativityQuests',
        'CompletedKillQuests',
        'CompletedExploreQuests',
        'CompletedMasteryQuests',
        'CompletedImmersionQuests',
        'CompletedReportQuests',
        'CompletedListenQuests',
        'CompletedExchangeQuests',
        'CompletedGiveQuests',
        'CompletedGoToQuests',
        'CompletedGatherQuests'
    ]

    available_quests = [col for col in quest_columns if col in dungeon_metadata_df.columns]

    if len(available_quests) == 0:
        print("Nenhuma coluna de quest encontrada")
        return pd.DataFrame()

    quest_names = {
        'CompletedKillQuests': 'combate',
        'CompletedExploreQuests': 'exploracao',
        'CompletedCreativityQuests': 'criatividade',
        'CompletedMasteryQuests': 'maestria',
        'CompletedImmersionQuests': 'imersao',
        'CompletedReportQuests': 'report',
        'CompletedListenQuests': 'listen',
        'CompletedExchangeQuests': 'troca',
        'CompletedGiveQuests': 'doacao',
        'CompletedGoToQuests': 'locomocao',
        'CompletedGatherQuests': 'coleta'
    }

    resultado = []

    for user in recommended_levels_df['User'].unique():
        user_recs = recommended_levels_df[recommended_levels_df['User'] == user].head(top_n).copy()

        user_recs = user_recs.merge(dungeon_metadata_df[['LevelName'] + available_quests], left_on='Item', right_on='LevelName', how='left')

        user_recs = user_recs.merge(ratings_df[ratings_df['User'] == user][['Item', 'Rating']], on='Item', how='left')

        if len(user_recs) == 0:
            continue

        user_profile = user_profile_df[user_profile_df['PlayerId'] == user]
        preform = user_profile['PreFormAnswers'].iloc[0] if len(user_profile) > 0 else 'N/A'

        rating_mean = user_recs['Rating'].mean()

        quest_means = {}
        for quest in available_quests:
            quest_means[quest] = user_recs[quest].mean()

        if len(quest_means) == 0:
            continue

        sorted_quests = sorted(quest_means.items(), key=lambda x: x[1], reverse=True)

        top3_quests = []
        for quest, score in sorted_quests[:3]:
            nome_legivel = quest_names.get(quest, quest.replace('Completed', '').replace('Quests', '').lower())
            top3_quests.append(f"{nome_legivel}({score:.2f})")

        recomendacao = f"top3: " + " > ".join(top3_quests)

        resultado.append({
            'User': user,
            'PreFormAnswers': preform,
            'rating_medio': round(rating_mean, 2),
            'preferencia_recomendada': recomendacao,
            'quest_primaria': quest_names.get(sorted_quests[0][0], sorted_quests[0][0]) if len(sorted_quests) > 0 else 'N/A',
            'score_primario': round(sorted_quests[0][1], 2) if len(sorted_quests) > 0 else 0,
            'top3_quests': top3_quests,
            'niveis_base': user_recs['Item'].tolist()
        })

    return pd.DataFrame(resultado)

def aggregate_recommendations_by_keys(recommended_levels_df, dungeon_metadata_df, top_n=5):
    """
    recommended_levels_df: DataFrame com colunas ['User', 'Item', 'Score'] onde Item = LevelName
    dungeon_metadata_df: DataFrame com colunas ['LevelName', 'TotalKeys']
    """

    resultado = []

    for user in recommended_levels_df['User'].unique():
        user_recs = recommended_levels_df[recommended_levels_df['User'] == user].head(top_n).copy()

        user_recs = user_recs.merge(dungeon_metadata_df[['LevelName', 'TotalKeys']], left_on='Item', right_on='LevelName', how='left')

        if len(user_recs) == 0:
            continue

        media_chaves = user_recs['TotalKeys'].mean()

        if media_chaves <= 3:
            recomendacao = "0-3 chaves"
        elif media_chaves <= 6:
            recomendacao = "4-6 chaves"
        elif media_chaves <= 9:
            recomendacao = "7-9 chaves"
        else:
            recomendacao = "10+ chaves"

        resultado.append({
            'User': user,
            'faixa_recomendada': recomendacao,
            'media_chaves': round(media_chaves, 1),
            'min_chaves': user_recs['TotalKeys'].min(),
            'max_chaves': user_recs['TotalKeys'].max(),
            'niveis_base': user_recs['Item'].tolist()
        })

    return pd.DataFrame(resultado)

In [77]:
def recomendando(method, user_item_matrix, user_similarity_df, item_similarity_df):
  # ------------- RECOMENDAR ITEM POR ITEM ------------#
      # esta comentado, pq nao fareirpos oturos casos
      if(method >=1 and method <=4):
        # Example recommendation
        similar_items = recommend_similar_items('A=4, C=3, I=2, M=1', item_similarity_df)
        print("\nItems similar to A=4, C=3, I=2, M=1:", similar_items)

      # Recomendar todos os itens por similarida itens
      print("--------------------6----------------------------")
      all_recommendations = {}

      for item in user_item_matrix.columns:
          all_recommendations[item] = recommend_similar_items(item, item_similarity_df)

      print(all_recommendations)

      # ------------- SIMILIARIDA USUARIO POR USUARIO ------------#
      print("--------------------7----------------------------")

      # print abaixo é pra ENTENDNEO O RESULTADO ACIMA
      print_user_ratings(user_item_matrix, user_similarity_df)

      # ------------- testando um usuario com mais de um nivel ------------#
      print("--------------------8----------------------------")
      #analyze_user_activity(user_item_matrix) # -> TODO; Descobri onde que tivrei e voltar

      # Recomendar para u1 baseado em u2
      print("--------------------9----------------------------")
      recommend_from_other_user(user_item_matrix.index[0], user_item_matrix.index[1], user_item_matrix)

      # Recomendar para u2 baseado em u1
      print("--------------------10----------------------------")
      recommend_from_other_user(user_item_matrix.index[1], user_item_matrix.index[0], user_item_matrix)

      # Escolher usuário
      print("--------------------13----------------------------")
      recommend_user_based_explain(user_item_matrix.index[0], user_item_matrix, user_similarity_df, top_n=5)

      # 🔥 MULTI-VIEW RECOMMENDATION (NOVO)
      print("--------------------14----------------------------")
      if method == 13:
          print("\n\n=========== MULTI VIEW ===========")
          multi_view_recommendation(_df, _df['PlayerId'].iloc[0])
      elif (method >= 14):
          print("\n=========== HYBRID RECOMMENDATION ===========")
          recommend_hybrid(user_item_matrix.index[0], user_item_matrix, item_similarity_df, user_similarity_df)

In [83]:
def collaborative_filtering(_df, method, teste, recomenda):
      print("-------------------- VISUALIZAÇÃO DOS DADOS ----------------------------")
      print(_df.shape)
      print(_df.head())
      print("-"*80)

      print("\nFrequência no dungeon_frame final:")
      print(_df['PreFormAnswers'].value_counts())

      # Número de usuários únicos pelo PlayerId
      num_unicos = _df['PlayerId'].nunique()
      print(f"\nNúmero de usuários únicos: {num_unicos}")

      print("-------------------- NORMALIZAÇÃO DOS DADOS ----------------------------")
      collab_df = build_collab_df(_df, method)
      test_df = build_collab_df(teste, method)

      print("\nMatriz para filtragem 10 :")
      print(test_df.head())

      print("\nMatriz para filtragem 1 :")
      print(collab_df.head())

      print("-------------------- CÁLCULOS ----------------------------")

      # Create user-item matrix
      user_item_matrix = collab_df.pivot_table(index='User', columns='Item', values='Rating', fill_value=0)
      test_matrix = test_df.pivot_table(index='User', columns='Item', values='Rating', fill_value=0)
      print(test_matrix.head())

      n_users = user_item_matrix.shape[0]
      n_items = user_item_matrix.shape[1]

      total_cells = n_users * n_items
      non_zero = (user_item_matrix > 0).sum().sum()

      sparsity = 1 - (non_zero / total_cells)

      print("\n===== SPARSITY =====")
      print("Users:", n_users)
      print("Items:", n_items)
      print("Total cells:", total_cells)
      print("Non-zero interactions:", non_zero)
      print("Sparsity:", sparsity)

      print("-------------------- VERIFICAÇÃO DOS DADOS ----------------------------")
      analyze_user_item_matrix(user_item_matrix)

      print("-------------------- SIMILIARIDADE ----------------------------")
      if(method % 2 == 0):
        item_similarity_df, user_similarity_df = similiarity_pearson(user_item_matrix)
      else:
        item_similarity_df, user_similarity_df = similiarity_cosine(user_item_matrix)

      if(recomenda):
        recomendando(method, user_item_matrix, user_similarity_df, item_similarity_df)


      # ---------------------------------------------
      # PREDIÇÃO APENAS NO TEST SET
      # ---------------------------------------------
      #TODO; PAREI AQUI - ORGANIZAR

      # 1. Garantir interseção estrutural
      print("TEST MATRIX SHAPE BEFORE:", test_matrix.shape)
      print("TRAIN MATRIX SHAPE:", user_item_matrix.shape)

      common_users = set(collab_df["User"]).intersection(test_df["User"])
      common_items = set(collab_df["Item"]).intersection(test_df["Item"])

      print("\nANTES DO FILTRO:")
      print("Test size:", len(test_df))
      print("Common users:", len(common_users))
      print("Common items:", len(common_items))

      test_df = test_df[
          test_df["User"].isin(common_users) &
          test_df["Item"].isin(common_items)
      ]

      print("\nDEPOIS DO FILTRO:")
      print("Test size:", len(test_df))

      if len(test_df) == 0:
          print("\n TEST DF VIROU VAZIO → pulando métricas")
          return


      print("TEST MATRIX SHAPE AFTER:", test_matrix.shape)


      all_items = user_item_matrix.columns

      recommendations_user = []

      for user in test_df["User"].unique():

          for item in all_items:

              score = predict_user_based(
                  user,
                  item,
                  user_item_matrix,
                  user_similarity_df
              )

              recommendations_user.append({
                  "User": user,
                  "Item": item,
                  "Score": score
              })
      recommendations_item = []

      for user in test_df["User"].unique():

          for item in all_items:

              score = predict_item_based(
                  user,
                  item,
                  user_item_matrix,
                  item_similarity_df
              )

              recommendations_item.append({
                  "User": user,
                  "Item": item,
                  "Score": score
              })
      rec_user_df = pd.DataFrame(recommendations_user)

      rec_user_df = rec_user_df.sort_values(
          ["User", "Score"],
          ascending=[True, False]
      )

      rec_item_df = pd.DataFrame(recommendations_item)

      rec_item_df = rec_item_df.sort_values(
          ["User", "Score"],
          ascending=[True, False]
      )

      predictions = []
      predictions_item = []

      for _, row in test_df.iterrows():

          user = row["User"]
          item = row["Item"]
          true_rating = row["Rating"]

          # ---------- USER-BASED ----------
          pred_user = predict_user_based(
              user,
              item,
              user_item_matrix,
              user_similarity_df
          )

          # ---------- ITEM-BASED ----------
          pred_item = predict_item_based(
              user,
              item,
              user_item_matrix,
              item_similarity_df
          )

          predictions.append({
              "User": user,
              "Item": item,
              "Rating": true_rating,
              "Prediction": pred_user
          })

          predictions_item.append({
              "User": user,
              "Item": item,
              "Rating": true_rating,
              "Prediction": pred_item
          })


      print("\n===== ITEM-BASED =====")

      pred_df = pd.DataFrame(predictions)
      pred_item_df = pd.DataFrame(predictions_item)

      print("\n===== DEBUG INTERSEÇÃO =====")
      print("Itens únicos preditos:", pred_df['Item'].nunique())
      print("Itens únicos no teste:", test_df['Item'].nunique())

      intersection = set(pred_df['Item']) & set(test_df['Item'])
      print("Interseção de itens:", len(intersection))
      print("Itens em comum:", intersection)

      print("\n===== DEBUG USUÁRIOS =====")
      print(test_df.groupby("User")["Item"].count())

      # remover NaN
      pred_df = pred_df.dropna()
      pred_item_df = pred_item_df.dropna()

      rmse = np.sqrt(mean_squared_error(pred_df["Rating"], pred_df["Prediction"]))
      mae  = mean_absolute_error(pred_df["Rating"], pred_df["Prediction"])
      print("Predições válidas:", len(pred_df))
      print("RMSE:", rmse)
      print("MAE:", mae)
      rmse_item = np.sqrt(mean_squared_error(pred_item_df["Rating"], pred_item_df["Prediction"]))
      mae_item  = mean_absolute_error(pred_item_df["Rating"], pred_item_df["Prediction"])
      print("RMSE item:", rmse_item)
      print("MAE item:", mae_item)
      precision = precision_at_k(pred_df, test_df, k=5, threshold=1.0)
      recall = recall_at_k(pred_df, test_df, k=5, threshold=1.0)

      precision_user = precision_at_k(rec_user_df, test_df, k=8, threshold=0.0)
      recall_user    = recall_at_k(rec_user_df, test_df, k=8, threshold=0.0)
      precision_item = precision_at_k(rec_item_df, test_df, k=8, threshold=0.0)
      recall_item    = recall_at_k(rec_item_df, test_df, k=8, threshold=0.0)

      print("Precision@5:", precision_user)
      print("Recall@5:", recall_user)
      print("Precision@5 item:", precision_item)
      print("Recall@5 item:", recall_item)

      hr_user = hit_rate_at_k(rec_user_df, test_df, k=8)
      mrr_user = mrr(rec_user_df, test_df)

      hr_item = hit_rate_at_k(rec_item_df, test_df, k=8)
      mrr_item = mrr(rec_item_df, test_df)

      print("HitRate@5:", hr_user)
      print("MRR:", mrr_user)
      print("HitRate@5:", hr_item)
      print("MRR:", mrr_item)


      if method >= 5 and method <= 8:
            if len(rec_user_df) > 0:
                quest_columns = [
                    'CompletedCreativityQuests',
                    'CompletedKillQuests',
                    'CompletedExploreQuests',
                    'CompletedMasteryQuests',
                    'CompletedImmersionQuests',
                    'CompletedReportQuests',
                    'CompletedListenQuests',
                    'CompletedExchangeQuests',
                    'CompletedGiveQuests',
                    'CompletedGoToQuests',
                    'CompletedGatherQuests'
                ]

                available_quests = [col for col in quest_columns if col in _df.columns]
                dungeon_quests = _df[['LevelName'] + available_quests].drop_duplicates()
                user_profiles = _df[['PlayerId', 'PreFormAnswers']].drop_duplicates()

                agregacao_quests = aggregate_recommendations_by_quest(rec_user_df, dungeon_quests, user_profiles, collab_df, top_n=5)

                print("\n=== RECOMENDACOES AGREGADAS POR PREFERENCIA DE QUESTS ===")
                if len(agregacao_quests) > 0:
                    print(agregacao_quests[['User', 'PreFormAnswers', 'rating_medio', 'preferencia_recomendada']].to_string())

                agregacao_inimigos = aggregate_recommendations_by_enemy_difficulty(rec_user_df, _df, user_profiles, collab_df, top_n=5)

                print("\n=== RECOMENDACOES AGREGADAS POR DIFICULDADE DE INIMIGOS ===")
                if len(agregacao_inimigos) > 0:
                    print(agregacao_inimigos[['User', 'PreFormAnswers', 'rating_medio', 'kill_rate_medio', 'recomendacao_inimigos']].to_string())

                agregacao_npcs = aggregate_recommendations_by_npcs(rec_user_df, _df, user_profiles, collab_df, top_n=5)

                print("\n=== RECOMENDACOES AGREGADAS POR NPCS ===")
                if len(agregacao_npcs) > 0:
                    print(agregacao_npcs[['User', 'PreFormAnswers', 'rating_medio', 'interacao_ratio', 'recomendacao_npcs']].to_string())

                agregacao_progresso = aggregate_recommendations_by_progress(rec_user_df, _df, user_profiles, collab_df, top_n=5)

                print("\n=== RECOMENDACOES AGREGADAS POR PROGRESSO (TESOUROS/TEMPO) ===")
                if len(agregacao_progresso) > 0:
                    print(agregacao_progresso[['User', 'PreFormAnswers', 'rating_medio', 'treasure_ratio', 'recomendacao_progresso']].to_string())

                agregacao_estrutura = aggregate_recommendations_by_structure(rec_user_df, _df, user_profiles, collab_df, top_n=5)

                print("\n=== RECOMENDACOES AGREGADAS POR ESTRUTURA (SALAS/CHAVES/FECHADURAS) ===")
                if len(agregacao_estrutura) > 0:
                    print(agregacao_estrutura[['User', 'PreFormAnswers', 'rating_medio', 'rooms_medio', 'recomendacao_estrutura']].to_string())

                return {
                    'rmse': rmse,
                    'mae': mae,
                    'rmse_item': rmse_item,
                    'mae_item': mae_item,
                    'precision_user': precision_user,
                    'recall_user': recall_user,
                    'hitrate_user': hr_user,
                    'mrr_user': mrr_user,
                    'recommendations': rec_user_df,
                    'quest_aggregations': agregacao_quests,
                    'enemy_aggregations': agregacao_inimigos,
                    'npcs_aggregations': agregacao_npcs,
                    'progress_aggregations': agregacao_progresso,
                    'structure_aggregations': agregacao_estrutura
                }


      return {
          'rmse': rmse,
          'mae': mae,
          'rmse_item': rmse_item,
          'mae_item': mae_item,
          'precision_user': precision_user,
          'recall_user': recall_user,
          'precision_item': precision_item,
          'recall_item': recall_item,
          'hitrate_user': hr_user,
          'mrr_user': mrr_user,
          'hitrate_item': hr_item,
          'mrr_item': mrr_item
      }


def run(_df, method, recomenda):
    # Organizando o dataset
    _df['PreFormAnswers'] = _df['PreFormAnswers'].apply(lambda x: 'A=' + str(x[0]) + ", C=" + str(x[1]) + ", I=" + str(x[2]) + ", M=" + str(x[3]))
    _df = split_preform_answers(_df)
    n = _df.shape[0]
    print("_df shape:", _df.shape)

    # ANTES DO SPLIT (ESSENCIAL)
    counts = _df.groupby("PlayerId").size()
    valid_users = counts[counts >= 2].index
    _df = _df[_df["PlayerId"].isin(valid_users)]

    # SPLIT
    train_df, test_df = leave_one_out_split(_df)

    # Normalizar
    numeric_cols = train_df.select_dtypes(include=["float64", "int64"]).columns
    cols_to_exclude = ["PlayerID", "PreFormAnswers", "LevelName"]
    numeric_cols = [c for c in numeric_cols if c not in cols_to_exclude]
    scaler = MinMaxScaler()
    train_df[numeric_cols] = scaler.fit_transform(train_df[numeric_cols])
    test_df[numeric_cols] = scaler.transform(test_df[numeric_cols])


    # Augmentation
    _dfnoise = augment_data_random(train_df, ['PreFormAnswers', 'A', 'C', 'I', 'M'], n)
    _dfscale = augment_dataframe_with_scaling(train_df, ['PreFormAnswers', 'A', 'C', 'I', 'M'], n, 1.05)
    _dfnoise2 = augment_data_random(train_df, ['PreFormAnswers', 'A', 'C', 'I', 'M'], 2*n)
    _dfscale2 = augment_dataframe_with_scaling(train_df, ['PreFormAnswers', 'A', 'C', 'I', 'M'], n, 1.05)
    _dfscale2 = augment_dataframe_with_scaling(train_df, ['PreFormAnswers', 'A', 'C', 'I', 'M'], n, 0.95)
    _dfmixed = augment_data_random(train_df, ['PreFormAnswers', 'A', 'C', 'I', 'M'], n)
    _dfmixed = pd.concat([_dfmixed, augment_dataframe_with_scaling(train_df, ['PreFormAnswers', 'A', 'C', 'I', 'M'], n, 1.05)], ignore_index=True)
    _dfmixed10 = augment_data_random(train_df, ['PreFormAnswers', 'A', 'C', 'I', 'M'], 6*n)
    _dfmixed10 = pd.concat([_dfmixed10, augment_dataframe_with_scaling(train_df, ['PreFormAnswers', 'A', 'C', 'I', 'M'], n, 1.05)], ignore_index=True)
    _dfmixed10 = pd.concat([_dfmixed10, augment_dataframe_with_scaling(train_df, ['PreFormAnswers', 'A', 'C', 'I', 'M'], n, 0.95)], ignore_index=True)
    _dfmixed10 = pd.concat([_dfmixed10, augment_dataframe_with_scaling(train_df, ['PreFormAnswers', 'A', 'C', 'I', 'M'], n, 1.10)], ignore_index=True)
    _dfmixed10 = pd.concat([_dfmixed10, augment_dataframe_with_scaling(train_df, ['PreFormAnswers', 'A', 'C', 'I', 'M'], n, 0.9)], ignore_index=True)

    dataframes = [#_df, _dfnoise, _dfscale, _dfnoise2, _dfscale2, _dfmixed,
                  _dfmixed10]
    experiment_names = [#'Baseline', '1xN', '1xS-1_05', '2xN', '2xS-1_05+0_95', '1xS-1_05+1xN',
                        '6xN+4xS-1_05+0_95+1_1+0_9']

    strategies = [#'Original', '1xNoise', '1xScale', '2xN', '2xScale', '1xN+1xS',
                  '6xN+4xS']

    # Quantos níveis diferentes cada usuário jogou
    levels_per_user = _df.groupby("PlayerId")["LevelName"].nunique()

    # Filtrar apenas quem jogou mais de 1 nível
    multi_level_users = levels_per_user[levels_per_user > 1]

    print("Usuários que jogaram mais de um nível:")
    print(multi_level_users)

    # Filtragem colaborativa
    collaborative_filtering(_dfmixed10, method, test_df, recomenda)


def main(method, recomenda):

    initialize_data_frame()
    dungeon_frame = get_dungeon_data_frame()
    user_frame = get_user_data_frame()

    if(method == 0):
      inspect_dataset(user_frame, dungeon_frame)
    else:
      dungeon_frame = pd.merge(dungeon_frame, user_frame[['PlayerId', 'PreFormAnswers']], on=['PlayerId'], how='left')
      dungeon_frame = drop_invalid_rows(dungeon_frame)
      dungeon_frame = get_percentages(dungeon_frame)
      run(dungeon_frame, method, recomenda)


In [84]:
main(0, False)

Combinações únicas:
[(1, 3, 2, 4), (2, 1, 3, 4), (2, 1, 4, 3), (3, 2, 1, 4), (3, 2, 4, 1), (4, 1, 3, 2), (4, 2, 1, 3), (4, 3, 2, 1)]

Quantidade de combinações únicas:
8

Frequência de cada combinação:
PreFormAnswers
(4, 2, 1, 3)    3
(3, 2, 4, 1)    3
(1, 3, 2, 4)    3
(3, 2, 1, 4)    2
(4, 3, 2, 1)    1
(4, 1, 3, 2)    1
(2, 1, 3, 4)    1
(2, 1, 4, 3)    1
Name: count, dtype: int64

Frequência no dungeon_frame final:
PreFormAnswers
[1, 3, 2, 4]    26
[4, 1, 3, 2]    25
[2, 1, 3, 4]    24
[4, 2, 1, 3]    18
[3, 2, 1, 4]    11
[3, 2, 4, 1]     9
[2, 1, 4, 3]     5
[4, 3, 2, 1]     1
Name: count, dtype: int64

Total de colunas: 57

  1. CompletedAchievementQuests
  2. CompletedCreativityQuests
  3. CompletedDamageQuests
  4. CompletedExchangeQuests
  5. CompletedExploreQuests
  6. CompletedGatherQuests
  7. CompletedGiveQuests
  8. CompletedGoToQuests
  9. CompletedImmersionQuests
 10. CompletedKillQuests
 11. CompletedListenQuests
 12. CompletedMasteryQuests
 13. CompletedQuests
 14. C

In [87]:
main(7, False) # 1 E 5 FOI, VER PQ 9 NAO FOI

/tmp/ipykernel_1142/1089917324.py:93: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  _data_frame.replace([np.inf, -np.inf], 0, inplace=True)


_df shape: (119, 67)
Usuários que jogaram mais de um nível:
PlayerId
53627948       4
78194049       3
81972292      11
121412094     12
282268519      4
1093542367    12
1153731241     4
1776427228     2
2079201442     4
2079942857    12
2096844718     3
Name: LevelName, dtype: int64
-------------------- VISUALIZAÇÃO DOS DADOS ----------------------------
(1710, 67)
                      PostFormAnswers  TotalEnemies  TotalMasteryQuests  MaxCombo  TotalExploreQuests  TotalNpcs  CompletedReportQuests  CompletedListenQuests  TotalDeaths  CompletedAchievementQuests  TotalGiveQuests  TotalAchievementQuests  TotalKeys  PlayerId  TotalLocks  KeysCollected  TotalExchangeQuests  TotalLostHealth  CompletedExploreQuests  CompletedExchangeQuests  TotalGoToQuests  RoomsEntered  CompletedDamageQuests  TotalAttempts  TotalCreativityQuests  TotalGatherQuests  NpcsInteracted  CompletedMasteryQuests                                       InputProfile  CompletedGiveQuests WeaponName  TotalImmersionQuest

/usr/local/lib/python3.12/dist-packages/numpy/_core/fromnumeric.py:3596: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
/usr/local/lib/python3.12/dist-packages/numpy/_core/_methods.py:138: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)



=== RECOMENDACOES AGREGADAS POR DIFICULDADE DE INIMIGOS ===
        User      PreFormAnswers  rating_medio  kill_rate_medio                                                           recomendacao_inimigos
0   0.000000  A=3, C=2, I=1, M=4          0.31             0.48  recomendar dungeons com MUITO POUCOS inimigos (usuario esta tendo dificuldade)
1   0.012023  A=3, C=2, I=1, M=4          0.77             0.59                            recomendar dungeons com quantidade MEDIA de inimigos
2   0.013872  A=4, C=1, I=3, M=2          0.80             0.76                        recomendar dungeons com MUITOS inimigos (kill_rate alto)
3   0.033175  A=4, C=2, I=1, M=3          0.80             0.85                        recomendar dungeons com MUITOS inimigos (kill_rate alto)
4   0.111902  A=1, C=3, I=2, M=4          0.24             0.36  recomendar dungeons com MUITO POUCOS inimigos (usuario esta tendo dificuldade)
5   0.508959  A=2, C=1, I=3, M=4          0.78             0.85            



## Atributos Estruturais
1. **TotalRooms** - número de salas
2. **TotalKeys** - número de chaves
3. **TotalLocks** - número de fechaduras
4. **UniqueRoomsEntered** - salas únicas visitadas
5. **RoomsEntered** - total de salas entradas (com revisitas)

## Atributos de Combate
6. **TotalEnemies** - número total de inimigos
7. **EnemiesKilled** - inimigos mortos
8. **MaxCombo** - maior combo
9. **TotalLostHealth** - vida perdida
10. **TotalDeaths** - número de mortes

## Atributos de Interação
11. **TotalNpcs** - número de NPCs
12. **NpcsInteracted** - NPCs com quem interagiu

## Atributos de Progresso
13. **TotalTreasure** - tesouros disponíveis
14. **TreasuresCollected** - tesouros coletados
15. **TimeToFinish** - tempo para finalizar

## Atributos de Missões (Quests)
16. **TotalAchievementQuests** - total de quests de conquista
17. **TotalCreativityQuests** - total de quests de criatividade
18. **TotalDamageQuests** - total de quests de dano
19. **TotalExploreQuests** - total de quests de exploração
20. **TotalGatherQuests** - total de quests de coleta
21. **TotalGiveQuests** - total de quests de entrega
22. **TotalGoToQuests** - total de quests de "ir até"
23. **TotalImmersionQuests** - total de quests de imersão
24. **TotalKillQuests** - total de quests de matar
25. **TotalListenQuests** - total de quests de ouvir
26. **TotalMasteryQuests** - total de quests de mestria
27. **TotalReadQuests** - total de quests de ler
28. **TotalReportQuests** - total de quests de relatório

## Métricas Calculadas
29. **LockUsage** - uso de fechaduras (LocksOpened/TotalLocks)
30. **KeyCollection** - coleta de chaves (KeysCollected/TotalKeys)
31. **MapCompletion** - exploração (UniqueRoomsEntered/TotalRooms)
32. **RoomRevisitRate** - taxa de revisita
33. **EnemyKillRate** - taxa de abate (EnemiesKilled/TotalEnemies)
34. **Success Rate** - taxa de sucesso (Wins/Attempts)



#TODO
- 1º FAZER RECOMENDACAO MULTI

-- recomendar level e explica - recomendado pq tem x salsa, filtrar

-- criar item composto ex: D: 10, C: 20, I: 4 - DIFICUA, CHAVE, INIMIGO... - FAZRE UM TIPO DE NOME PARA CASOS - AI USAR COLUNA

-- ANTES DE IR RECOMENDADO PERIMEIRO JUNTAR

- 2º Normalizar os dados (1 a 0)
- 3º PCA (usar todos os dados e depois usar o pca para ver quais sao inuties, assim tirar eles)
- 4º fazer as analises como dito no artigo (eu tenho que faze-las
- 5º Organizar o código mais ainda

#add no artigo
- poderia usar o perfil para melhora a recomendação
- recomendaco a aprtir de todos usuarios? ex.: um grupo aprova isso e vc ta no grupo, entao vc vai gostar



